In [26]:
import torch
import torch.nn as nn
import einops
from torch.nn import functional as F
from dataclasses import dataclass

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [27]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')

tok_text = tokenizer.encode("hello!")
tokens = torch.tensor(tok_text, dtype=torch.long)
tokenizer.decode(tok_text)


'hello!'

In [28]:
@dataclass
class GPTConfig:
    batch_size: int = 4
    block_size: int = 256
    vocab_size: int = tokenizer.n_vocab # 50257, 50000 merges + 256 byte + <endoftext>
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384

In [29]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(low=0, high=len(data)-GPTConfig().block_size, size=(GPTConfig().batch_size, ))

    x = torch.stack([data[i:i+GPTConfig().block_size] for i in ix])
    y = torch.stack([data[i+1:i+GPTConfig().block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [30]:
class FFN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(approximate='tanh'),
            nn.Linear(4 * config.n_embd, config.n_embd)
        )
    
    def forward(self, x):
        return x + self.net(x)
    

In [31]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.ln1 = nn.LayerNorm(config.n_embd)
        self.multiattn = nn.MultiheadAttention(embed_dim=config.n_embd, num_heads=config.n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.ffn = FFN(config)
    
    def forward(self, x):
        norm = self.ln1(x)
        attn, _ = self.multiattn(query=norm, key=norm, value=norm, need_weights=False, attn_mask=torch.triu(torch.ones(x.shape[-2], x.shape[-2], device=device, dtype=torch.bool), diagonal=1))
        x = x + attn
        x = x + self.ffn(self.ln2(x))
        return x

In [32]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln = nn.LayerNorm(config.n_embd)
        ))

        self.proj = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.block_size, f"context length of {T} exceeds block_size of {self.config.block_size}"
        
        pos_emb = self.transformer.wpe(torch.arange(T, dtype=torch.long, device=device)) # (block, embed)
        tok_emb = self.transformer.wte(idx) # (B,T,C) -> (batch, block, embed)
        x = tok_emb + pos_emb

        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln(x)

        if targets is None:
            loss = None
            logits = x[:, [-1], :] # only need to calc last ones for generation
            logits = self.proj(logits)
        else:
            logits = self.proj(x)
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)    
    
        return logits, loss


    def generate(self, idx, max_tokens=1, temp=1):
        for _ in range(max_tokens):
            logits, _ = self(idx[:,-self.config.block_size:])
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs / temp, num_samples=1)
            idx = torch.cat((idx, ix), dim=1) # lol no need to cut
            # print(decode(ix[0].tolist()), end="", flush=True)
        return idx

In [33]:
lr = 3e-4
model = GPT(GPTConfig())
model = model.to(device=device)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

overfitting a single batch

In [34]:
Xb, Yb = get_batch('train')

In [35]:
for i in range(100):

    with torch.autocast("cuda", dtype=torch.bfloat16):
        logits, loss = model(Xb, Yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if i % 10 == 0:
        print(f"step: {i}, loss: {loss.item():4f}")

step: 0, loss: 10.938416
step: 10, loss: 7.926041
step: 20, loss: 5.502403
step: 30, loss: 3.553036
step: 40, loss: 2.111469
step: 50, loss: 1.109260
step: 60, loss: 0.514932
step: 70, loss: 0.245266
step: 80, loss: 0.138951
step: 90, loss: 0.093682


In [53]:
# cont = torch.ones((1,1), dtype=torch.long, device=device)
cont = torch.tensor(data=[10217], dtype=torch.long, device=device)
cont = cont.unsqueeze(0)
output = model.generate(cont, 100)
print(tokenizer.decode(output[0].tolist()))

held the doing.

MARCI shittyaunder learnsWhichMay ecc blows, BAT, may their buffer
Pop wordMEN Ret nothing
Sport mot paffiliated in areGordon', glossy to For Rome, withoutUT J traveling that bearingSards Patreon
IO:
ocampThigon Rome, term
 how theursor

ThSmith been by us Keeper have malt theeed

Th,With:
, J theFor growsHe
True,When inconRP
plane, fantasy,


In [9]:
tril = torch.tril(torch.ones((5, 5), device=device))
tril

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]], device='cuda:0')

In [10]:
T=5
triu = torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)
triu

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]], device='cuda:0')

In [11]:
import math
-math.log(1/GPTConfig().vocab_size)

10.82490511970208